# Measure the memory used by a decoder’s KV cache

_Authored by: Codex, on behalf of [Seongeun So](https://github.com/sungeuns), maintainer of Foundation Model Engineering._

A serving worker has 120 MiB left for attention keys and values. Can it take four requests with 512-token prompts and room for 256 more tokens each? The prompts fit. The continuations change the answer.

We will get the missing number, **bytes per stored token**, directly from a pretrained model. We'll measure the tensors after a padded prefill, watch them grow one position at a time, and check that reusing the cache preserves next-token logits. You can run everything with a CPU, a Python notebook with PyTorch installed, and internet access for the first model download (about 270 MB). The experiment was tested with Python 3.12, PyTorch 2.15.0 (CPU) and Transformers 5.24.1. In float32 the model parameters alone use roughly 514 MiB.

In [1]:
%pip install -q "transformers==5.24.1"

## Read the checkpoint geometry

[SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M) uses grouped-query attention. Several query heads read the same keys and values. Pin both model and tokenizer to the same revision. Float32 and evaluation mode make the CPU comparison below easy to reproduce.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache

model_id = "HuggingFaceTB/SmolLM2-135M"
revision = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=revision, dtype=torch.float32, attn_implementation="eager"
).eval().cpu()
cfg = model.config
layers = cfg.num_hidden_layers
kv_heads = cfg.num_key_value_heads
head_dim = cfg.head_dim
value_bytes = next(model.parameters()).element_size()
bytes_per_position = 2 * layers * kv_heads * head_dim * value_bytes
print(f"{layers} layers; {cfg.num_attention_heads} query heads; {kv_heads} KV heads")
print(f"{head_dim} values per head; {value_bytes} bytes per value")
print(f"{bytes_per_position:,} bytes per stored position per request")

30 layers; 9 query heads; 3 KV heads
64 values per head; 4 bytes per value
46,080 bytes per stored position per request


The factor of two counts K and V. For **B** requests with **T** stored positions each, full-context attention in this model uses `B × T × bytes_per_position` bytes of logical cache contents. Using the nine query heads instead of the three KV heads would triple the estimate.

## Measure the first forward pass

These prompts have different lengths. We left-pad them to form one rectangular tensor, using the attention mask to indicate which positions contain real text. We assign position IDs from the unpadded lengths so that each sequence starts at position zero.

In [3]:
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
prompts = ["Water boils at", "When a kettle of water starts to boil, the steam"]
encoded = tokenizer(prompts, return_tensors="pt", padding=True)
ids, mask = encoded.input_ids, encoded.attention_mask
batch, width = ids.shape

def position_ids(mask):
    return (mask.cumsum(dim=-1) - 1).clamp_min(0)

with torch.inference_mode():
    output = model(
        input_ids=ids, attention_mask=mask, position_ids=position_ids(mask),
        use_cache=True, logits_to_keep=1,
    )

cache = output.past_key_values
assert isinstance(cache, DynamicCache)
print("Unpadded prompt lengths:", mask.sum(dim=1).tolist())
print("First layer's key shape:", tuple(cache.layers[0].keys.shape))

Unpadded prompt lengths: [3, 11]
First layer's key shape: (2, 3, 11, 64)


Each full-attention layer stores two tensors with shape `[batch, KV heads, positions, head dimension]`. Count their elements and read the actual element size. Comparing this measurement with the prediction catches mistakes such as counting query heads or assuming the checkpoint's on-disk dtype is the runtime cache dtype.

In [4]:
def measure_cache(cache, positions):
    assert len(cache.layers) == layers
    measured = 0
    for layer in cache.layers:
        for tensor in (layer.keys, layer.values):
            assert tensor.shape == (batch, kv_heads, positions, head_dim)
            measured += tensor.numel() * tensor.element_size()
    expected = batch * positions * bytes_per_position
    assert measured == expected, (measured, expected)
    return measured

prefill_bytes = measure_cache(cache, width)
padding_positions = batch * width - mask.sum().item()
print(f"Prompt cache: {prefill_bytes:,} bytes")
print(f"Of these, {padding_positions * bytes_per_position:,} bytes occupy padding positions")

Prompt cache: 1,013,760 bytes
Of these, 368,640 bytes occupy padding positions


The mask prevents attention to padded tokens. The dense cache still has a slot for each of those positions. If you change the prompts, try to predict the allocation from their *longer* tokenized length before running the measurement again.

## Append four tokens and check the logits

The prefill returns logits for the first new token. Selecting that token does not yet extend the cache. The cache grows when we feed it back into the model. At each step we give the cached call just the new token and the attention mask for the entire sequence so far. As a control, we separately recompute the whole sequence without caching and compare the complete next-token logit vectors. We run exactly four steps to make the comparison reproducible.

In [5]:
next_ids = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
with torch.inference_mode():
    for step in range(1, 5):
        ids = torch.cat([ids, next_ids], dim=1)
        mask = torch.cat([mask, torch.ones_like(next_ids)], dim=1)
        positions = position_ids(mask)
        cached = model(
            input_ids=next_ids, attention_mask=mask, position_ids=positions[:, -1:],
            past_key_values=cache, use_cache=True, logits_to_keep=1,
        )
        cache = cached.past_key_values
        reference = model(
            input_ids=ids, attention_mask=mask, position_ids=positions,
            use_cache=False, logits_to_keep=1,
        )
        torch.testing.assert_close(cached.logits, reference.logits, rtol=1e-4, atol=1e-4)
        measured = measure_cache(cache, width + step)
        delta = (cached.logits - reference.logits).abs().max().item()
        print(f"+{step}: {measured:,} cache bytes; largest logit difference {delta:.2e}")
        next_ids = cached.logits[:, -1, :].argmax(dim=-1, keepdim=True)

+1: 1,105,920 cache bytes; largest logit difference 2.05e-05
+2: 1,198,080 cache bytes; largest logit difference 1.96e-05
+3: 1,290,240 cache bytes; largest logit difference 2.10e-05
+4: 1,382,400 cache bytes; largest logit difference 2.57e-05


Different matrix shapes can produce small floating-point differences even when the cached computation is correct. The byte count, on the other hand, should increase by exactly two sequence positions per iteration, one for each request.

## Decide how many requests to admit

Return to the hypothetical worker with 120 MiB available **for KV tensors after reserving all other memory**. Each request starts with 512 tokens. Budget conservatively for 256 additional *stored* positions per request. Here we calculate this workload from the measured geometry without allocating a larger batch.

In [6]:
MiB = 1024**2
budget = 120 * MiB
requests, prompt, continuation = 4, 512, 256
for stage, length in [("prompt", prompt), ("reserved", prompt + continuation)]:
    per_request = length * bytes_per_position
    print(
        f"{stage}: {requests} requests use {requests * per_request / MiB:g} MiB; "
        f"{budget // per_request} requests fit"
    )

prompt: 4 requests use 90 MiB; 5 requests fit
reserved: 4 requests use 135 MiB; 3 requests fit


Four prompts occupy 90 MiB. Four fully grown requests require 135 MiB, so only three fit the reservation. A scheduler that admits all four needs a plan if they grow together, such as preempting a request. An output that ends early may never fill its reservation.

This estimate covers one CPU copy of logical K/V tensor elements for a dense, full-context dynamic cache. Process memory also includes model weights, activations, temporary concatenation buffers and allocator overhead. Static caches can reserve unused positions. Sliding windows cap some layers' histories. Paged runtimes can round to blocks or share common prefixes. Inspect the tensors or block allocator for the runtime you intend to size.

### Sources

- [Pinned SmolLM2 model configuration](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/blob/93efa2f097d58c2a74874c7e644dbc9b0cee75a2/config.json)
- [How caching works in Transformers](https://huggingface.co/docs/transformers/en/cache_explanation) and [the cache implementation used here](https://github.com/huggingface/transformers/blob/v5.24.1/src/transformers/cache_utils.py)
- [Foundation Model Engineering, KV cache management](https://sungeuns.github.io/foundation-model-engineering/chapter-12/kv-cache-management/#kv-cache-capacity), the source lesson for the admission calculation